# Categorical simulation

This document includes a well-calibrated simulation study for testing the sBayes clustering algorithm. We simulate parameters by drawing samples from the prior distribution, generate synthetic data from these, and pass the data to the sBayes algorithm to infer the simulated parameters. We then evaluate the calibration of the inference procedure by comparing the inferred posterior distributions to the true parameter values.

We use the Gemini LLM to create an empty structure of the synthetic data using the following prompt:

Create a CSV with 20 rows and the following columns:

    name: use any first names you can think of
    id: abbreviate the first names to a unique id with three upper case letters
    x: a random longitude
    y: a random latitude
    confounder_1: assign each row randomly to A or B
    f1: keep empty
    f2: keep empty
    ...
    f30: keep empty

In [1]:
from sbayes.experiment_setup import Experiment
from sbayes.load_data import Data as Structure, Data
from sbayes.mcmc_setup import MCMCSetup
from sbayes.sampling.loggers import write_samples, CategoricalFeatures
from numpyro.infer import Predictive
from pathlib import Path
import jax.random as random
import numpy as np
import pandas as pd
import shutil
import os
import re
import matplotlib.pyplot as plt
from typing import Tuple, Dict, Optional, List, Any
from numpy.typing import NDArray

Next, we set up the model using the ``config.yaml file``. This file specifies the number of simulated clusters and confounders, and defines the data type for each feature. In this experiment, all features are categorical with two, three, or four discrete states.

In [2]:
# Initialize the experiment

experiment = Experiment(
    config_file="config.yaml",
    experiment_name="categorical",
)

# Enabling sampling from the prior
experiment.config.model.sample_from_prior = True

# Load the model structure (number of observations, variables, confounders, clusters)
structure = Structure.from_experiment(experiment)

# Set up Model
setup = MCMCSetup(structure, experiment)
model = setup.model.get_model

# We don't need the usual subfolders for this simulation
shutil.rmtree(experiment.path_results)


/home/peter/Desktop/sBayes/sBayes/sbayes/config/config.py:298: UserWarning: No `type` defined for `ClusterEffectConfig`. Using `uniform` as a default.
  warnings.warn(
Experiment: categorical
File location for results: /home/peter/Desktop/sBayes/sBayes/experiments/simulation_categorical/sims/categorical
Start time and date: 16:33:49 04.08.2025


DATA IMPORT
##########################################
20 objects with 30 features read from /home/peter/Desktop/sBayes/sBayes/experiments/simulation_categorical/template_data/features.csv.
Categorical[2]: 10 feature(s) with 200 NA value(s).
Categorical[3]: 10 feature(s) with 200 NA value(s).
Categorical[4]: 10 feature(s) with 200 NA value(s).
/home/peter/Desktop/sBayes/sBayes/sbayes/config/config.py:298: UserWarning: No `type` defined for `DirichletPriorConfig`. Using `uniform` as a default.
  warnings.warn(


We draw 100 independent sets of parameters from the prior distribution. For each set, we generate a corresponding synthetic dataset.

In [3]:
rng_key = random.PRNGKey(0)
num_samples = 100

# Set up Predictive to draw from prior
predictive = Predictive(model, num_samples=num_samples)

# Sample parameters and synthetic data from prior
prior_samples = predictive(rng_key)

We define functions to write the sampled parameters and corresponding synthetic data to file.

In [4]:
def prepare_folder(
        path: Path,
        idx: int
) -> Tuple[Path, Path]:
    """
    creates a folder structure for a simulation run.
    Args:
        path (Path): Base directory where the simulation folders will be created.
        idx (int): Index of the simulation sample (used to name the folder).

    Returns:
        Tuple[Path, Path]: Paths to the parameters folder, and data folder.
    """
    i_folder = path / f"sim_{idx}"
    i_folder.mkdir(parents=False, exist_ok=True)

    p_folder = i_folder / "sim_params"
    p_folder.mkdir(parents=False, exist_ok=True)

    d_folder = i_folder / "sim_data"
    d_folder.mkdir(parents=False, exist_ok=True)

    return p_folder, d_folder

def write_data(
    partitions: list,
    sample: Dict[str, np.ndarray],
    features_csv: pd.DataFrame,
    base_path: Path
) -> None:
    """
    updates features CSV file with simulation data for a given sample index and saves it.

    Args:
        partitions (Any): List containing the different partitions, e.g., Categorical[2]
        sample (Dict[str, np.ndarray]): Dictionary of prior sample keyed by feature name.
        features_csv (pd.DataFrame): DataFrame containing the non-simulated data, otherwise empty
        base_path (Path): Path to the folder where the updated features CSV will be saved.

    this function:
        - Updates the features_csv DataFrame with the new simulated data.
        - Writes the updated DataFrame to `features.csv` in the data folder.
    """
    for ft in partitions:
        # index 0 is correct, t ith sample was sliced earlier while retaining dimensionality
        sim = sample[f"x_{ft.name}"][0]

        # Rename categorical indices to state names
        if isinstance(ft, CategoricalFeatures):
            col_indices = np.arange(sim.shape[1])[None, :]
            sim = ft.state_names[col_indices, sim]

        df = pd.DataFrame(sim, columns=ft.names)
        features_csv[ft.names] = features_csv[ft.names].astype(str)
        features_csv.update(df)

    features_csv.to_csv(base_path / "features.csv", index=False)


We write the sampled parameters and corresponding synthetic data to file.


In [ ]:
empty_features_csv = pd.read_csv(experiment.config.data.features)
results_folder = experiment.config.results.path

# Write samples and data to file
for s in range(num_samples):

    params_folder, data_folder = prepare_folder(results_folder, s)

    i_sample = {k: v[s:s+1] for k, v in prior_samples.items()}

    write_samples(run=0, base_path=params_folder,
                  samples=i_sample,
                  data=structure, model=setup.model)

    write_data(partitions=structure.features.partitions,
               sample=i_sample,features_csv=empty_features_csv.copy(deep=True),
               base_path=data_folder)

Next, for each of the 100 synthetic datasets, we perform inference to recover the corresponding set of sampled parameters.

In [ ]:
# Run inference
for s in range(num_samples):

    experiment.config.model.sample_from_prior = False
    experiment.config.data.features = results_folder / f"sim_{s}/sim_data/features.csv"
    experiment.path_results = results_folder / f"sim_{s}/results"
    experiment.path_results.mkdir(parents=False, exist_ok=True)

    # Load the data
    data = Data.from_experiment(experiment)
    # Set up Model
    mcmc = MCMCSetup(data, experiment)
    mcmc.sample(resume=False)


We define functions to read the posterior distribution over the parameters for each inference run.

In [4]:
def sort_by_number(name: str) -> int:
    """
    Extracts and returns the numeric suffix from a string separated by underscores.
    Used to sort folder or file 'sim_1', 'sim_10', etc.,
    by extracting the number at the end for proper numerical ordering.

    Args:
        name (str): A string that ends with an underscore followed by a number.

    Returns:
        int: The numeric suffix extracted from the string.
    """
    return int(name.split('_')[-1])

def read_parameters(base_path: Path, k: int = None):
    """Read the simulated and inferred parameters

     Args:
         base_path: The folder name with the simulated parameters
         k: number of clusters in the model, if None it will be inferred
     Returns:
         simulated and inferred parameters
     """

    sims = os.listdir(base_path)

    params = dict()

    # Open the simulation runs in the sims folder one by one
    for i in sorted(sims, key=sort_by_number):
        params[i] = dict()
        if k:
            stats = f"stats_K{k}_0.txt"
            params[i]['simulated'] = pd.read_csv(base_path / i / "sim_params" /stats,
                                                 delimiter="\t")
            params[i]['inferred'] = pd.read_csv(base_path / i / f"results/K{k}/"/stats,
                                                delimiter="\t")

    return params


For each of the 100 inference runs, we read in the posterior distribution over the parameters.

In [8]:
#results_folder = experiment.config.results.path
parameters = read_parameters(results_folder, k=2)


Next, we define functions to plot the inferred parameters against the simulated (true) values.

In [16]:
def find_conf_prefix(name: str, confounders: List[str]) -> Tuple[str, str]:
    """
    Extracts the confounder prefix from the given name.

    Args:
        name: The input string expected to start with a confounder prefix.
        confounders: A list of known confounder name strings.

    Returns:
        A tuple containing:
            - The matching confounder string.
            - The remaining portion of the name after removing the prefix.

    Raises:
        ValueError: If no confounder prefix matches the start of the name.
    """
    for conf in confounders:
        prefix = f"{conf}_"
        if name.startswith(prefix):
            return conf, name.removeprefix(prefix)
    raise ValueError(f"No matching confounder found in '{name}'")

def find_group_prefix(name: str, groups: List[str]) -> Tuple[str, str]:
    """
    Extracts the group prefix from the given name.

    Args:
        name: The input string expected to start with a group prefix.
        groups: A list of known group name strings.

    Returns:
        A tuple containing:
            - The matching group string.
            - The remaining portion of the name after removing the prefix.

    Raises:
        ValueError: If no group prefix matches the start of the name.
    """
    for g in groups:
        prefix = f"{g}_"
        if name.startswith(prefix):
            return g, name.removeprefix(prefix)
    raise ValueError(f"No matching group found in '{name}'")

def find_feature_prefix(name, features):
    """
    Extracts the feature prefix from the given name.

    Args:
        name: The input string expected to start with a feature prefix.
        features: A list of known feature name strings.

    Returns:
        A tuple containing:
            - The matching feature string.
            - The remaining portion of the name after removing the prefix.

    Raises:
        ValueError: If no feature prefix matches the start of the name.
    """
    for f in features:
        prefix = f"{f}_"
        if name.startswith(prefix):
            return f, name.removeprefix(prefix)
    raise ValueError(f"No matching feature found in '{name}'")

def find_areal_prefix(name: str) -> Tuple[int, str]:
    """
    Extracts the areal cluster number and remaining string from the given name.

    The function expects the name to start with a prefix matching the pattern "areal_a<digits>_".
    It extracts the number after 'a', increments it by 1, and returns it along with the
    remaining part of the name after removing the matched prefix.

    Args:
        name: The input string expected to start with an areal prefix, e.g. "areal_a12_..."iWe read in the posterior distribution for each inferred


    Returns:
        A tuple containing:
            - The incremented areal cluster number (int).
            - The remaining portion of the name after removing the areal prefix.

    Raises:
        ValueError: If the input string does not match the expected pattern.
    """
    match_obj = re.match(r"(areal_a\d+_)", name)
    if not match_obj:
        raise ValueError(f"Name '{name}' does not start with a valid areal prefix")

    match = match_obj.group(1)

    number_match = re.search(r"a(\d+)", match)
    if not number_match:
        raise ValueError(f"Could not extract number from areal prefix '{match}'")

    number = number_match.group(1)

    return int(number) + 1, name.removeprefix(match)

def find_title(
    name: str,
    confounders: Dict[str, Any],
    feature_names: List[str]
) -> Optional[str]:
    """
    Generates a descriptive title based on the given parameter name.

    The function parses the input name string to identify confounder prefixes, groups,
    features, and components, or areal cluster prefixes, returning a human-readable
    description.

    Args:
        name: The parameter name string to parse.
        confounders: A dictionary mapping confounder names to confounder objects.
                     Each confounder object must have a 'group_names' attribute,
                     which is a list of group name strings.
        feature_names: A list of valid feature name strings.

    Returns:
        A descriptive string summarizing the feature/component/group information,
        or None if the name does not match any expected pattern.

    """
    confounder_names = confounders.keys()
    confounder_prefixes = tuple(f"{conf}_" for conf in confounder_names)
    w_confounder_prefixes = tuple(f"w_{conf}_" for conf in confounder_names)

    if name.startswith(confounder_prefixes):

        conf, remaining = find_conf_prefix(name, confounder_names)
        group, remaining = find_group_prefix(remaining,
                                             confounders[conf].group_names)
        feature, component = find_feature_prefix(remaining, feature_names)
        return f"Feature {feature}, component {component} in {conf}, group {group}"

    elif name.startswith("areal_"):
        cluster, remaining = find_areal_prefix(name)
        feature, component = find_feature_prefix(remaining, feature_names)
        return f"Feature {feature}, component {component} in cluster {cluster}"

    elif name.startswith(w_confounder_prefixes):
        conf, feature = find_conf_prefix(name.removeprefix("w_"), confounder_names)
        return f"Weights for feature {feature} and {conf}"

    elif name.startswith("w_areal_"):
        feature = name.removeprefix("w_areal_")
        return f"Cluster weights for feature {feature}"
    else:
        return None

def plot_simulated_against_inferred(
    simulated: NDArray[np.floating],
    inferred: NDArray[np.floating],
    title: Optional[str] = None,
    ax: Optional[plt.Axes] = None
) -> None:
    """
    Plot simulated values on the x-axis against inferred distributions on the y-axis.

    Each row in `inf` corresponds to one simulated value in `sim`. The function checks
    whether each simulated value falls within the 5th–95th percentile of its corresponding
    inferred distribution and color-codes the points accordingly.

    Parameters:
        simulated (NDArray[np.floating]): Array of shape (n,), containing simulated values.
        inferred (NDArray[np.floating]): Array of shape (n, m), containing inferred distributions.
        title (Optional[str]): Optional title for the plot.
        ax (Optional[plt.Axes]): Matplotlib Axes to plot on. Defaults to current axis.

    Returns:
        None
    """
    if ax is None:
        ax = plt.gca()

    low_perc = np.percentile(inferred, 2.5, axis=1)
    high_perc = np.percentile(inferred, 97.5, axis=1)
    in_perc = (low_perc < simulated) & (simulated < high_perc)

    min_val = np.min([inferred.min(), simulated.min()])
    max_val = np.max([inferred.max(), simulated.max()])

    for i, sim in enumerate(simulated):
        color = 'grey' if in_perc[i] else 'red'
        ax.plot([sim] * inferred.shape[1], inferred[i],
                'o', markersize=1, alpha=0.1, color=color)

    ax.axline((0, 0), slope=1, color='black')
    ax.text(
        0.99, 0.01,
        f"{np.sum(in_perc)} / {len(simulated)}",
        ha='right',
        va='bottom',
        transform=ax.transAxes
    )

    ax.set_xlabel('Simulated')
    ax.set_ylabel('Estimated')
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)
    ax.grid(False)

    if title:
        ax.set_title(title)


We plot the simulated (true) parameters against the inferred posteriors. We expect that, on average, the true parameter values fall within the 95% credible intervals of the posterior distributions approximately 95% of the time.

In [17]:

column_names_sim = next(iter(parameters.values()))['simulated'].columns.tolist()

for n in column_names_sim:

    if n in ['Sample']:
        pass
    else:
        p_sim = np.array([v['simulated'][n][0] for v in parameters.values()])
        p_inf = np.array([v['inferred'][n] for v in parameters.values()])
        title_plot = find_title(n, structure.confounders, structure.features.names,)
        plot_simulated_against_inferred(simulated=p_sim, inferred=p_inf,
                                        title=title_plot)
        plot_folder = results_folder.parent / "plots"
        plot_folder.mkdir(parents=False, exist_ok=True)
        plt.savefig(plot_folder / f"{n}.png")
        plt.close()
